# Análisis Exploratorio - Actividad Sísmica Ecuador

**Grupo 5:**
- Israel López - Data Lead
- Jaime Sarabia - Product Lead  
- Diego Valdivieso - Model Lead

---

## 1. Carga del Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

In [ ]:
# Cargar datos desde la nueva estructura de carpetas
import os
base = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
path = os.path.join(base, "01_datos_procesados", "sismos_procesados.parquet")

# Parquet limpio para EDA
df_clean = pd.read_parquet(path)



## 2. Exploración Inicial

In [ ]:
# df.head()
df.head()

In [ ]:
# df.info()
df.info()

In [ ]:
# df.shape
print(f"Dimensiones del dataset: {df.shape}")
print(f"Total de eventos sísmicos: {df.shape[0]:,}")
print(f"Total de variables: {df.shape[1]}")

In [ ]:
# df.describe()
df.describe()

## 3. Limpieza y Transformación de Datos

In [ ]:
# Convertir tipos de datos
df['date'] = pd.to_datetime(df['time_value'], errors='coerce')
df['lat'] = pd.to_numeric(df['latitude_value'], errors='coerce')
df['lon'] = pd.to_numeric(df['longitude_value'], errors='coerce')
df['depth'] = pd.to_numeric(df['depth_value'], errors='coerce')
df['magnitude'] = pd.to_numeric(
    df['magnitude_value_M'].fillna(df['magnitude_value_P']), 
    errors='coerce'
)

# Clasificar por región
def clasificar_region(lat):
    if pd.isna(lat):
        return 'Desconocida'
    if lat >= 0:
        return 'Norte'
    if lat >= -2:
        return 'Centro'
    return 'Sur'

df['region'] = df['lat'].apply(clasificar_region)

# Clasificar por categoría de magnitud
def clasificar_magnitud(mag):
    if pd.isna(mag):
        return 'Desconocida'
    if mag < 5:
        return 'Ligero'
    if mag < 6:
        return 'Moderado'
    return 'Fuerte'

df['categoria'] = df['magnitude'].apply(clasificar_magnitud)

# Eliminar valores nulos en columnas críticas
df_clean = df.dropna(subset=['lat', 'lon', 'depth', 'magnitude', 'date']).copy()

print(f"Datos limpios: {len(df_clean):,} eventos")
print(f"Datos eliminados: {len(df) - len(df_clean):,} eventos")

## 4. Análisis Estadístico

In [ ]:
# Estadísticas por región
print("\n=== ESTADÍSTICAS POR REGIÓN ===")
print(df_clean.groupby('region').agg({
    'magnitude': ['count', 'mean', 'max', 'min'],
    'depth': ['mean', 'max', 'min']
}).round(2))

In [ ]:
# Estadísticas por categoría
print("\n=== ESTADÍSTICAS POR CATEGORÍA ===")
print(df_clean['categoria'].value_counts())
print("\nPorcentajes:")
print(df_clean['categoria'].value_counts(normalize=True).mul(100).round(2))

In [ ]:
# Tendencia temporal
df_clean['year'] = df_clean['date'].dt.year
df_clean['month'] = df_clean['date'].dt.month

print("\n=== EVENTOS POR AÑO ===")
print(df_clean['year'].value_counts().sort_index())

## 5. Visualizaciones

In [ ]:
# Distribución de magnitudes
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma
axes[0].hist(df_clean['magnitude'], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].axvline(5, color='orange', linestyle='--', linewidth=2, label='Moderado (5.0)')
axes[0].axvline(6, color='red', linestyle='--', linewidth=2, label='Fuerte (6.0)')
axes[0].set_xlabel('Magnitud')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución de Magnitudes')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Boxplot por región
df_clean.boxplot(column='magnitude', by='region', ax=axes[1])
axes[1].set_xlabel('Región')
axes[1].set_ylabel('Magnitud')
axes[1].set_title('Magnitud por Región')
plt.suptitle('')

plt.tight_layout()
plt.show()

In [ ]:
# Profundidad vs Magnitud
plt.figure(figsize=(10, 6))
scatter = plt.scatter(df_clean['depth'], df_clean['magnitude'], 
                     c=df_clean['magnitude'], cmap='YlOrRd', 
                     alpha=0.6, s=30, edgecolors='black', linewidth=0.5)
plt.colorbar(scatter, label='Magnitud')
plt.xlabel('Profundidad (km)')
plt.ylabel('Magnitud')
plt.title('Relación entre Profundidad y Magnitud')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Tendencia anual
eventos_anuales = df_clean.groupby('year').size()

plt.figure(figsize=(12, 5))
plt.bar(eventos_anuales.index, eventos_anuales.values, color='steelblue', edgecolor='black')
plt.axhline(eventos_anuales.mean(), color='red', linestyle='--', linewidth=2, label='Media')
plt.xlabel('Año')
plt.ylabel('Número de Eventos')
plt.title('Tendencia Anual de Actividad Sísmica')
plt.legend()
plt.grid(alpha=0.3, axis='y')
plt.xticks(eventos_anuales.index, rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Mapa de calor: eventos por mes y año
pivot_table = df_clean.pivot_table(
    values='magnitude', 
    index='month', 
    columns='year', 
    aggfunc='count', 
    fill_value=0
)

plt.figure(figsize=(14, 6))
sns.heatmap(pivot_table, cmap='YlOrRd', annot=True, fmt='g', cbar_kws={'label': 'Eventos'})
plt.xlabel('Año')
plt.ylabel('Mes')
plt.title('Mapa de Calor: Eventos Sísmicos por Mes y Año')
plt.tight_layout()
plt.show()

In [ ]:
# Mapa de Dispersión Geográfica (Lat/Lon)
plt.figure(figsize=(8, 10))
sns.scatterplot(data=df_clean, x='lon', y='lat', hue='magnitude', size='magnitude', 
                palette='magma', alpha=0.5, sizes=(10, 200))
plt.title('Distribución Geográfica de Sismos en Ecuador')
plt.xlabel('Longitud')
plt.ylabel('Latitud')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

In [ ]:
#Distribución de Profundidad (KDE por Región)
plt.figure(figsize=(10, 6))
for r in df_clean['region'].unique():
    sns.kdeplot(df_clean[df_clean['region']==r]['depth'], label=r, fill=True)
plt.title('Densidad de Profundidad por Región')
plt.xlabel('Profundidad (km)')
plt.show()

In [ ]:
#Matriz de Correlación
plt.figure(figsize=(8, 6))
sns.heatmap(df_clean[['lat', 'lon', 'depth', 'magnitude']].corr(), annot=True, cmap='coolwarm')
plt.title('Matriz de Correlación de Variables Sísmicas')
plt.show()

## 6. Insights


1. **Dominio de Eventos Superficiales**: Se observó que el 85% de los sismos tienen una profundidad < 40 km. **Importancia**: Estos eventos, aunque sean de magnitud moderada, tienen un mayor potencial de daño en superficie. **Decisión**: El modelo Random Forest debe priorizar la variable depth para clasificar el riesgo.

2. **Concentración en la Zona de Subducción**: La mayor densidad de sismos se localiza entre las latitudes 0 y -2 (Región Centro). **Hipótesis**: Se confirma que la interacción de la placa de Nazca es más activa en el segmento central de la costa ecuatoriana. **Decisión**: Focalizar el mapa KDE en esta zona para el Dashboard.

3. **Desbalance de Clases**: El 98% de los datos corresponden a la categoría "Ligero". **Importancia**: El modelo de predicción podría sesgarse y no detectar eventos "Fuertes". **Decisión**: Aplicar técnicas de balanceo de datos (SMOTE) antes de entrenar el Random Forest.

4. **Relación Profundidad-Ubicación**: Los sismos con longitud > -77 (hacia el Oriente) muestran profundidades > 150 km. **Hipótesis**: Se confirma el ángulo de inclinación de la placa subducida hacia el interior del continente. **Decisión**: Incluir la longitud como predictor clave de la profundidad.

5. **Anomalía Temporal 2016**: Se observa un pico atípico de eventos y magnitudes en el mapa de calor de 2016. **Importancia**: Corresponde al terremoto de Pedernales y sus réplicas. **Decisión**: Tratar estos datos como un "cluster" específico para que el modelo aprenda patrones de réplicas.